# M4.3: sparse vs dense retrieval baselines

This notebook compares the frozen M4.2 E5 dense baseline with an evaluation-only BM25 lexical baseline. It does not combine rankings. If a CUDA failure occurred in this runtime, use **Runtime → Disconnect and delete runtime**, reopen the notebook, and Run All from a fresh session.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace this URL.
REPOSITORY_REF = 'main'  # Branch, tag, or commit to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6'
RUNTIME_METADATA = {'torchVersion': torch.__version__, 'transformersVersion': transformers.__version__, 'sentenceTransformersVersion': sentence_transformers.__version__, 'cudaDevice': GPU_NAME}

repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale modules are loaded. Restart the runtime, then Run All.')

GPU: CPU
torch: 2.11.0+cpu
transformers: 4.57.6
sentence-transformers: 5.6.0


In [3]:
import pandas as pd
from evals.src.dataset import load_dataset
from evals.src.embedding_eval import SentenceTransformerEmbeddingAdapter, embedding_model_registry, frozen_production_chunks
from evals.src.retrieval_eval import run_bm25_baseline, run_dense_baseline, save_retrieval_results

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/retrieval_eval_v1.json')
chunks = frozen_production_chunks(dataset)  # Generated once with 350/500/40 and reused by both retrievers.
e5_spec = embedding_model_registry()['multilingual_e5_large_instruct']
assert e5_spec.model_id == 'intfloat/multilingual-e5-large-instruct'
adapter = SentenceTransformerEmbeddingAdapter(e5_spec)
try:
    dense_result = run_dense_baseline(dataset, chunks=chunks, adapter=adapter)
finally:
    adapter.release()
sparse_result = run_bm25_baseline(dataset, chunks=chunks)
results = [dense_result, sparse_result]
save_retrieval_results(results, dataset_version=dataset.version, output_dir=ROOT / 'evals/results/retrieval', runtime_metadata=RUNTIME_METADATA)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [4]:
rows = []
for result in results:
    row = {'Retriever': result.retriever, **result.metrics}
    row['TR MRR'] = result.by_language.get('tr', {}).get('mrr', 0.0)
    row['EN MRR'] = result.by_language.get('en', {}).get('mrr', 0.0)
    row['Morphology MRR'] = result.by_category.get('morphology_heavy', {}).get('mrr', 0.0)
    row['TerminologyMismatch MRR'] = result.by_category.get('terminology_mismatch', {}).get('mrr', 0.0)
    row['HardParaphrase MRR'] = result.by_category.get('hard_paraphrase', {}).get('mrr', 0.0)
    row.update(result.efficiency)
    rows.append(row)
comparison = pd.DataFrame(rows)
display(comparison)

category_rows = []
for category in sorted(set(dense_result.by_category) | set(sparse_result.by_category)):
    category_rows.append({'Category': category, 'Dense MRR': dense_result.by_category.get(category, {}).get('mrr', 0.0), 'BM25 MRR': sparse_result.by_category.get(category, {}).get('mrr', 0.0)})
display(pd.DataFrame(category_rows))

for metric, label in [('recall_at_5', 'Recall@5'), ('recall_at_10', 'Recall@10'), ('mrr', 'MRR'), ('ndcg_at_10', 'nDCG@10'), ('hit_rate_at_5', 'HitRate@5'), ('required_block_coverage_at_5', 'BlockCoverage@5'), ('required_block_coverage_at_10', 'BlockCoverage@10')]:
    winners = comparison.loc[comparison[metric] == comparison[metric].max(), 'Retriever'].tolist()
    print(f'Best {label}: {winners}')

for category in ['morphology_heavy', 'terminology_mismatch', 'hard_paraphrase', 'near_negative', 'same_topic_competitor']:
    dense_mrr = dense_result.by_category.get(category, {}).get('mrr', 0.0)
    sparse_mrr = sparse_result.by_category.get(category, {}).get('mrr', 0.0)
    winner = 'tie' if dense_mrr == sparse_mrr else ('dense' if dense_mrr > sparse_mrr else 'BM25')
    print(f'{category}: dense MRR={dense_mrr:.4f}, BM25 MRR={sparse_mrr:.4f}, winner={winner}')

,Retriever,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10,TR MRR,EN MRR,Morphology MRR,TerminologyMismatch MRR,HardParaphrase MRR,query_embedding_seconds,query_retrieval_seconds,chunk_count,index_build_seconds
0,Dense — intfloat/multilingual-e5-large-instruct,0.958333,1.000000,0.964286,0.870040,0.851617,0.958333,1.000000,0.940476,0.799603,1.0,0.513889,1.000000,41.733809,0.174197,12,NaN
1,Sparse — BM25,0.910714,0.958333,0.916667,0.844577,0.825343,0.910714,0.958333,0.938492,0.750661,1.0,0.479167,0.925926,NaN,0.003359,12,0.000856


,Category,Dense MRR,BM25 MRR
0,cross_paragraph,0.652778,0.472222
1,factual,0.884259,0.900463
2,hard_paraphrase,1.000000,0.925926
3,heading_dependent,1.000000,0.916667
4,morphology_heavy,1.000000,1.000000
5,multi_section,0.916667,1.000000
6,near_negative,0.805556,0.722222
7,paraphrase,0.916667,0.916667
8,same_topic_competitor,0.805556,0.847222
9,terminology_mismatch,0.513889,0.479167


Best Recall@5: ['Dense — intfloat/multilingual-e5-large-instruct']
Best Recall@10: ['Dense — intfloat/multilingual-e5-large-instruct']
Best MRR: ['Dense — intfloat/multilingual-e5-large-instruct']
Best nDCG@10: ['Dense — intfloat/multilingual-e5-large-instruct']
Best HitRate@5: ['Dense — intfloat/multilingual-e5-large-instruct']
Best BlockCoverage@5: ['Dense — intfloat/multilingual-e5-large-instruct']
Best BlockCoverage@10: ['Dense — intfloat/multilingual-e5-large-instruct']
morphology_heavy: dense MRR=1.0000, BM25 MRR=1.0000, winner=tie
terminology_mismatch: dense MRR=0.5139, BM25 MRR=0.4792, winner=dense
hard_paraphrase: dense MRR=1.0000, BM25 MRR=0.9259, winner=dense
near_negative: dense MRR=0.8056, BM25 MRR=0.7222, winner=dense
same_topic_competitor: dense MRR=0.8056, BM25 MRR=0.8472, winner=BM25


In [5]:
from google.colab import files

files.download(
    "/content/prompt-generator-rag/evals/results/retrieval/sparse_dense_results_v1.json"
)

files.download(
    "/content/prompt-generator-rag/evals/results/retrieval/sparse_dense_results_v1.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>